# 面向长对话的贝叶斯自适应上下文选择 · Release
**ARIN7101 · 五人协作工作本 · release-1 · 2026-11-05**

研究问题：在受控合成长对话中，利用问题特征的 Thompson Sampling 能否提高“正确性 − 输入成本”的平均效用？

这是小组的唯一工作 notebook。实现参考已完成的可行性版本，**默认 demo；小组人工审计、正式实验、slides 和视频仍待完成**。自动运行成功不会把成员任务标为 done，也不会自动代替任何人签名。既有可行性结果不是本组新正式实验的结论。

先读 [README](README.md) → [协作与验收说明](docs/TEAM_WORKFLOW.md) → 本人的模块。参考完整本保留在 `reference/`，不作为运行入口。

| 成员 | 本节工作顺序 | 必须交付 | 接收与复核 |
|---|---|---|---|
| [M1](#member-m1) 数据 | 理解生成规则 → 追踪10题真值 → 填逐题记录 | review.json逐题M1签名、M1说明 | M2复核；M4冻结 |
| [M2](#member-m2) 策略 | 核对四臂 → 预算/时序 → 逐题交叉审查 → 解释反例 | 逐题M2签名、策略与成本说明 | M3接口复核；M4接收 |
| [M3](#member-m3) 方法 | 手算Beta → MAP/Hessian → 反馈隔离 → 核查输出 | 方法说明、method_checks.json解读 | M4复核 |
| [M4](#member-m4) 实验 | 收齐审查 → 开发基线 → 冻结 → 正式运行 → 对话级统计 | 完整manifest与结果路径、M4解释 | M3复核统计；M5接收 |
| [M5](#member-m5) 汇报 | 先搭slides → 读取指定归档 → 七图 → 逐图解读 → 录制提交 | M5说明、slides、视频、交付清单 | 全员复核 |

主线顺序：共享配置 → M1数据 → M2策略及审计导出 → M3检查 → **人工交接点** → M4 → M5。M1的人工审计依赖M2导出的材料，不能在生成策略前提前完成。


## 0 · 全员：首次运行与路径约定
Python 3.11 或 3.12；无需GPU、API密钥、外部数据和真实LLM。请先保存 notebook，再 Restart Kernel and Run All。

```bash
python -m pip install -r requirements.txt
python scripts/run_notebook.py --stage development
python scripts/run_notebook.py --stage all --mode demo
python -m jupyter lab
```

命令行 `development` 只执行共享配置及M1–M3，适合准备人工审查；`all` 执行完整demo与七图；正式运行操作见M4。全部可写路径从项目根目录解析，脚本从任意工作目录调用均可。

| 路径 | 内容 | 是否提交Git |
|---|---|---|
| `team_workspace/team_progress.csv` | 姓名、状态、证据、接收人 | 是，人工更新 |
| `team_workspace/review.json` | 从本次review_template复制并填写的真实审查 | 是，M4合并 |
| `team_workspace/formal_runs.json` | 正式运行登记，含失败/已查看情况 | 是，M4维护 |
| `outputs/demo/<run_id>/experiment/` | 开发/演示数据、CSV、审计模板、配置、manifest | 否 |
| `outputs/formal/seed_<seed>/experiment/` | 冻结后的正式归档，禁止覆盖 | 否，另交完整归档 |
| `outputs/<mode>/<run_id>/visualizations/<render_id>/` | 七组PNG/SVG、制图CSV、图表manifest | 否 |
| `outputs/<mode>/<run_id>/execution/` | 执行日志、带输出的notebook | 否 |
| `deliverables/` | 经复核的说明、slides PDF；视频仅留链接 | 是，大文件另交 |

不要在根目录写CSV/PNG，不要将机器生成的输出覆盖人工记录。每个导出单元使用统一的 `OUT` 或 `M5_DIR`；其具体文件名列在代码、[输出契约](docs/OUTPUT_CONTRACT.md)及manifest中。代码/配置变更后，旧审查哈希失效，需要重新审查。


In [ ]:
import os, re, json, hashlib, time, platform, textwrap, gzip, shutil, sys
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
from functools import lru_cache
import numpy as np
import pandas as pd
import scipy, sklearn, matplotlib
from scipy.special import expit
from scipy.optimize import minimize
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

# 唯一配置入口。交互运行默认demo；正式seed只读取团队审查文件。
def find_root():
    candidates = [Path(os.environ['ARIN_PROJECT_ROOT'])] if os.environ.get('ARIN_PROJECT_ROOT') else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate/'ARIN7101_Context_Bandit_Release_Notebook.ipynb').is_file() and (candidate/'requirements.txt').is_file():
            return candidate.resolve()
    raise FileNotFoundError('请从仓库目录启动Jupyter，或设置ARIN_PROJECT_ROOT。')

ROOT = find_root()
NOTEBOOK_PATH = ROOT/'ARIN7101_Context_Bandit_Release_Notebook.ipynb'
TEAM_DIR = ROOT/'team_workspace'
MODE = os.environ.get('ARIN_MODE', 'demo')
assert MODE in ['demo', 'formal']
review_file = TEAM_DIR/'review.json'
TEAM_REVIEW = json.loads(review_file.read_text(encoding='utf-8')) if review_file.exists() else {}
FINAL_SEED = TEAM_REVIEW.get('formal_seed')
if MODE == 'formal':
    assert type(FINAL_SEED) is int and 1_000_000 <= FINAL_SEED < 2**32-200, 'M4须先在review.json填写未使用的formal_seed。'
    # 完整可行性版本已使用的测试seed，不能作为新确认性证据。
    assert not (set(range(FINAL_SEED, FINAL_SEED+30)) | set(range(FINAL_SEED+100, FINAL_SEED+130))) & (set(range(20261105,20261135)) | set(range(20261205,20261235))), '与已知可行性实验seed重叠。'
CFG = dict(version='release-1', T=300, n_dev=3, n_eval=6 if MODE=='demo' else 30,
           u=.25, q=.02, pA=.90, pB=.10, conflict_fraction=.25, lam=.5, cref=512,
           gamma_grid=[.9,.97,.995,1.], dev_seed=130000,
           eval_seed=730000 if MODE=='demo' else FINAL_SEED,
           algorithm_seeds=[17,37], bootstrap_seed=2026, nboot=2000, refit_every=5,
           prior_precision=1., feature_names=['intercept','explicit_project','its'],
           budgets={'recent_short':128,'recent_long':512,'tfidf':256,'resolved_tfidf':256})
SCENARIOS = ['stable','abrupt']; ARMS = list(CFG['budgets'])
STAMP = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_ID = os.environ.get('ARIN_RUN_ID', STAMP) if MODE=='demo' else f'seed_{FINAL_SEED}'
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID), 'run_id只能含字母、数字、下划线和连字符。'
RUN_ROOT = ROOT/'outputs'/MODE/RUN_ID
OUT = RUN_ROOT/'experiment'
OUT.mkdir(parents=True, exist_ok=False)  # 任何模式都不覆盖已有实验
(OUT/'checkpoints').mkdir()
SYSTEM = 'Answer the current value using the context. Later statements override earlier ones. Resolve its using the latest focus. Return JSON {"value":"..."}; use UNKNOWN if unsupported.'
@lru_cache(maxsize=50000)
def units(s): return len(re.findall(r'\w+|[^\w\s]',s))
def prompt(q,ctx): return SYSTEM+'\nContext:\n'+'\n'.join(m['text'] for m in ctx)+'\nQuestion: '+q
def digest_file(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def save_json(name, value):
    (OUT/name).write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False)+'\n',encoding='utf-8')
source_document = json.loads(NOTEBOOK_PATH.read_text(encoding='utf-8'))
CODE_TEXT = '\n\n'.join(''.join(c['source']) for c in source_document['cells'] if c['cell_type']=='code')
CODE_HASH = hashlib.sha256(CODE_TEXT.encode()).hexdigest()
for c in source_document['cells']:
    if c['cell_type']=='code': c['outputs']=[]; c['execution_count']=None
save_json('notebook_snapshot.ipynb',source_document)
START = time.perf_counter()
CHECKS = {}
shutil.copy2(ROOT/'requirements.txt', OUT/'requirements.txt')
print('Mode:',MODE,'| regex proxy units, NOT LLM tokens | Output:',OUT.resolve())
print('数据/模型无需下载。人工进度不自动更新；提交Git前清空notebook输出。')
display(pd.read_csv(TEAM_DIR/'team_progress.csv').fillna(''))


<a id="member-m1"></a>
## M1 · 数据、问题定义与逐题审计
**目标：** 能从公开历史独立指出题目的对象、属性、最新证据和旧值，解释为什么长对话需要选择上下文。

1. 运行共享配置和本节，读 `generate`、`public_view`；画出“历史 → 当前问题 → 真值”的例子。核对 stable 的近期题概率0.9、abrupt在`t=150`由0.9变0.1。
2. 核查 `development_data_profile.csv` 与 `data_dictionary.json`；解释6个项目×4种属性、多版本事实与300题之间的相关性。
3. **等M2导出审计文件后**，打开同一次运行的 `audit_examples.json`（完整证据）与 `audit_compact.json`（阅读辅助），逐题追踪focus/its、对象、属性、最新证据ID、gold、过期值。不要只抄自动摘要。
4. 在共享 `review.json` 的10个reviews逐行填写 `m1_reviewer`、`notes`、`verdict`，让M2填写独立复核签名。发现问题填fail，修复后重跑development并重新绑定哈希。
5. 完成 `deliverables/M1_notes.md`，把具体run路径及审查文件提交M2/M4；接收人确认后再将进度改为done。

**输出：** `OUT/development_dialogues.jsonl.gz`、`development_data_profile.csv`、`data_dictionary.json`。自动解析器核查不代替人工签名。


In [ ]:
ALIASES={'dataset':'data source','language':'coding language','metric':'evaluation measure','deadline':'due date'}
ATTRS=list(ALIASES)
BOILER='Please use this setting in subsequent work and keep other project requirements unchanged.'
def normalize(s):
    for a,b in ALIASES.items(): s=s.replace(b,a)
    return s
FACT=re.compile(r'project (P\d+): the (dataset|language|metric|deadline) is (V\d+)')
FOCUS=re.compile(r'Focus is project (P\d+)')
def parse_fact(text):
    m=FACT.search(normalize(text)); return m.groups() if m else None
def target(q,ctx):
    p=re.search(r'P\d+',q)
    if p: project=p.group()
    else:
        f=[FOCUS.search(m['text']).group(1) for m in ctx if FOCUS.search(m['text'])]
        project=f[-1] if f else None
    qn=normalize(q); attr=next((a for a in ATTRS if re.search(r'\b'+a+r'\b',qn)),None)
    return project,attr

def generate(seed,scenario,T=None,conflict_fraction=None,explicit_recent=False,no_synonyms=False):
    rng=np.random.default_rng(seed); T=T or CFG['T']; hist=[]; truth={}; versions={}; latest={}; probes=[]
    projects=[f'P{seed*10+i}' for i in range(6)]; serial=0
    def append(text): hist.append({'id':len(hist),'text':text}); return len(hist)-1
    def fact(p,a,alias=False,new=True):
        nonlocal serial
        if new:
            serial+=1; v=f'V{seed*100000+serial}'; truth[p,a]=v; versions.setdefault((p,a),[]).append(v)
        else:v=truth[p,a]
        name=ALIASES[a] if alias and not no_synonyms else a
        latest[p,a]=append(f'For project {p}: the {name} is {v}. '+BOILER)
    cf=CFG['conflict_fraction'] if conflict_fraction is None else conflict_fraction
    for p in projects:
        for a in ATTRS:
            conflict=rng.random()<cf
            for _ in range(9 if conflict else 1):fact(p,a)
            if conflict:fact(p,a,alias=True)
    state=0
    for t in range(T):
        if scenario=='abrupt':state=int(t>=T//2)
        elif scenario=='markov' and t>0 and rng.random()<CFG['q']:state=1-state
        p=str(rng.choice(projects)); a=str(rng.choice(ATTRS))
        append(f'Focus is project {p}. We are reviewing its latest requirements.')
        fact(p,a,alias=True,new=rng.random()<CFG['u'])
        append('The next meeting will review progress and remaining questions.')
        recent=rng.random()<(CFG['pA'] if state==0 else CFG['pB'])
        qp=p if recent else str(rng.choice([z for z in projects if z!=p]))
        qa=a if recent else str(rng.choice(ATTRS))
        term=ALIASES[qa] if recent and not no_synonyms else qa
        ref=qp if not recent or explicit_recent else 'its'
        q=f'What is {ref} current {term}?'
        probes.append(dict(t=t,question=q,end=len(hist),gold=truth[qp,qa],
                           evidence=latest[qp,qa],old_values=versions[qp,qa][:-1].copy(),
                           qtype='recent' if recent else 'historical',state='A' if state==0 else 'B'))
    return dict(seed=seed,scenario=scenario,history=hist,probes=probes)
def public_view(ep,probe):
    return {'question':probe['question'],'history':ep['history'][:probe['end']]}

DEV=[generate(CFG['dev_seed']+100*s+i,sc) for s,sc in enumerate(SCENARIOS) for i in range(CFG['n_dev'])]
print('Development:',len(DEV),'dialogues;',CFG['T'],'questions/dialogue')


In [ ]:
# 与回答器独立实现的文本审计；避免用同一个函数同时生成“答案”和“证明”。
def independent_trace(question, history):
    explicit = re.search(r'\bP[0-9]+\b',question)
    focus = None
    term_map = {'dataset':'dataset','data source':'dataset','language':'language',
                'coding language':'language','metric':'metric','evaluation measure':'metric',
                'deadline':'deadline','due date':'deadline'}
    values = {}; evidence = {}; versions = {}
    for msg in history:
        text = msg['text']
        if text.startswith('Focus is project '): focus = text.split()[3].rstrip('.')
        if text.startswith('For project ') and ': the ' in text:
            project, rest = text[len('For project '):].split(': the ',1)
            label, tail = rest.split(' is ',1)
            key = (project,term_map[label]); value = tail.split('.',1)[0]
            values[key] = value; evidence[key] = msg['id']
            if value not in versions.setdefault(key,[]): versions[key].append(value)
    project = explicit.group() if explicit else focus
    question_term = question.split(' current ',1)[1].rstrip('?')
    key = (project,term_map[question_term])
    return dict(project=project, attribute=key[1], gold=values.get(key,'UNKNOWN'),
                evidence=evidence.get(key), old_values=versions.get(key,[])[:-1], focus=focus)

def audit_episode_truth(ep):
    for pr in ep['probes']:
        h = ep['history'][:pr['end']]
        check = independent_trace(pr['question'],h)
        assert check['gold']==pr['gold'] and check['evidence']==pr['evidence']
        assert check['old_values']==pr['old_values']
        assert pr['evidence']<pr['end'] and [m['id'] for m in h]==list(range(len(h)))
        assert (check['project']==check['focus']) == (pr['qtype']=='recent')
    return len(ep['probes'])

def save_episodes(episodes, name):
    # 存整条历史+截断索引，避免为每题重复存储完整历史。
    with gzip.open(OUT/name,'wt',encoding='utf-8') as f:
        for ep in episodes: f.write(json.dumps(ep,ensure_ascii=False)+'\n')

CHECKS['m1_development_independent_truth_questions'] = sum(audit_episode_truth(ep) for ep in DEV)
assert generate(CFG['dev_seed'],'stable') == DEV[0]
save_episodes(DEV,'development_dialogues.jsonl.gz')
data_rows = [dict(seed=ep['seed'],scenario=ep['scenario'],t=p['t'],qtype=p['qtype'],
                 state=p['state'],has_old=bool(p['old_values']),history_messages=p['end'])
             for ep in DEV for p in ep['probes']]
DATA_PROFILE = pd.DataFrame(data_rows)
profile = DATA_PROFILE.groupby(['scenario','state']).agg(
    questions=('t','size'), recent_fraction=('qtype',lambda s:(s=='recent').mean()),
    old_version_fraction=('has_old','mean'), mean_history_messages=('history_messages','mean'))
profile.to_csv(OUT/'development_data_profile.csv'); display(profile)
save_json('data_dictionary.json',{'generator':'synthetic-template-v2 (same mechanism as v4)',
    'history':'ordered public messages; each probe sees history[:end]',
    'evaluator_only':['gold','evidence','old_values','qtype','state','scenario'],
    'question_unit':'one query; online feedback immediate and noiseless',
    'statistical_unit':'independent dialogue; 300 dependent queries per dialogue',
    'external_data_required':False,'test_seed':CFG['eval_seed']})
print('M1 independent development truth audit:',CHECKS)


<a id="member-m2"></a>
## M2 · 四臂策略、成本与M1审计交接
**输入：** M1的DEV与公开问题/历史。**交付接收人：** M1（审计材料）、M3（接口）、M4（可冻结策略）。

1. 对照下表阅读 `select_context` / `rule_reader`，确认四臂共享回答器，近期窗口连续，检索结果按时间排序且无未来消息。
2. 运行本节，检查预算、完整/空历史、时序断言；读 `development_strategy_by_query.csv`，区分accuracy、coverage、outdated和总输入成本。
3. 把 `OUT/review_template.json` **首次复制**到 `team_workspace/review.json`；已有人工记录时先核对code/audit/protocol哈希，禁止直接覆盖。M1逐题填写后，由M2核对10题并填写 `m2_reviewer` 和复核说明。
4. 用 `strategy_boundary_cases.json` 写至少三个具体反例，注明seed/t/arm；解释为何取到事实不一定回答正确、为何不能把proxy成本称为实际token费用。
5. 填写 `deliverables/M2_notes.md` 和review中的M2 approval，把接口与检查证据交M3/M4。

**接口固定：** `public_view -> {question, history}`；`select_context(view, arm) -> 按id升序的消息列表`；`features(question) -> 3维向量`。不得把gold/qtype/state传入学习器。修改接口需同时更新文档并通知接收人。


### 3A · M2：把“准与省”写成可测量定义
**目标：** 用同一规则回答器公平比较上下文策略；分清候选上下文预算、总输入成本与构造耗时。

| 策略 | 操作 | 上下文预算 |
|---|---|---:|
| recent_short | 从最新消息倒序取最多3条，超预算即停 | 128 |
| recent_long | 连续取近期消息，超预算即停 | 512 |
| tfidf | 原问题词面检索，取最多8个候选，再按时间排序 | 256 |
| resolved_tfidf | 从公开历史消解its并归一化同义词，再检索；指代题另加最新focus | 256 |

预算单位是正则表达式计数 `regex proxy units`。resolved_tfidf可能在8个检索候选之外再加1条focus；最终仍服从256预算。

规则回答器在所选上下文里识别项目与属性，采用时间上最后一条可解析事实；没有支持则返回 `UNKNOWN`。正确性 $Y\in\{0,1\}$，总输入成本 $C$ 包含系统指令、上下文和问题：

$$U=Y-0.5C/512.$$

**关键边界：** 先构造所有候选以获知各臂成本。成本不是实际token、费用或推理延迟；候选构造耗时单独报告。主实验不模拟真实LLM调用。

**现在做：** 跟踪 `pack → select_context → rule_reader`；解释为什么检索中旧版本、同义词或缺失focus可能导致错误。下一格完成自动检查后，继续§3C人工抽查。


In [ ]:
def pack(history,indices,budget,stop_on_fail=False):
    # Newlines have zero regex units; summing whole-message costs is exact for this counter.
    chosen=[];used=0
    for i in indices:
        i=int(i);c=units(history[i]['text'])
        if used+c<=budget:chosen.append(i);used+=c
        elif stop_on_fail:break
    return [history[i] for i in sorted(chosen)]
def select_context(view,arm):
    h=view['history']; q=view['question']; budget=CFG['budgets'][arm]
    if not h: return []
    if arm.startswith('recent'):
        ids=list(range(len(h)-1,-1,-1))
        if arm=='recent_short': ids=ids[:3]
        return pack(h,ids,budget,stop_on_fail=True)
    docs=[m['text'] for m in h]
    if arm=='resolved_tfidf':
        p,a=target(q,h); q=f'{p or ""} {a or ""}'
        docs=[normalize(s) for s in docs]
    vector=TfidfVectorizer(); X=vector.fit_transform(docs)
    scores=(X@vector.transform([q]).T).toarray().ravel()
    ids=sorted(range(len(h)),key=lambda i:(scores[i],i),reverse=True)[:8]
    if arm=='resolved_tfidf' and not re.search(r'P\d+',view['question']):
        anchors=[i for i,m in enumerate(h) if FOCUS.search(m['text'])]
        if anchors:ids=[anchors[-1]]+[i for i in ids if i!=anchors[-1]]
    return pack(h,ids,budget)
def rule_reader(q,ctx):
    p,a=target(q,ctx); found={}
    for m in ctx:
        f=parse_fact(m['text'])
        if f:found[f[:2]]=f[2]
    return found.get((p,a),'UNKNOWN')
def features(q):
    return np.array([1.,float(bool(re.search(r'P\d+',q))),float('its' in q.split())])
def evaluate_episode(ep):
    rows=[]
    for pr in ep['probes']:
        view=public_view(ep,pr)
        assert set(view)=={'question','history'}
        assert rule_reader(view['question'],view['history'])==pr['gold'], 'full history audit'
        assert rule_reader(view['question'],[])=='UNKNOWN', 'empty context audit'
        for arm in ARMS:
            start=time.perf_counter(); ctx=select_context(view,arm); elapsed=time.perf_counter()-start
            assert units('\n'.join(m['text'] for m in ctx)) <= CFG['budgets'][arm], 'context budget'
            assert [m['id'] for m in ctx] == sorted(set(m['id'] for m in ctx)), 'unique chronological context'
            assert all(m['id']<pr['end'] for m in ctx), 'no future evidence'
            ans=rule_reader(view['question'],ctx)
            rows.append(dict(seed=ep['seed'],scenario=ep['scenario'],t=pr['t'],arm=arm,
                question=pr['question'],y=int(ans==pr['gold']),cost=units(prompt(pr['question'],ctx)),
                outdated=int(ans in pr['old_values']),coverage=int(pr['evidence'] in [m['id'] for m in ctx]),
                state=pr['state'],qtype=pr['qtype'],construction_ms=elapsed*1000))
    return pd.DataFrame(rows)

print('正在计算开发集四策略结果；TF-IDF逐题构造可能需要数分钟。')
DEV_OUT=pd.concat([evaluate_episode(e) for e in DEV],ignore_index=True)
DEV_OUT['utility']=DEV_OUT.y-CFG['lam']*DEV_OUT.cost/CFG['cref']
state_table=DEV_OUT.groupby(['state','arm']).utility.mean().unstack().reindex(columns=ARMS)
audit=DEV_OUT.groupby(['qtype','arm'])[['y','coverage','outdated','cost']].mean()
assert audit.loc[('recent','recent_short'),'coverage']>.95
print('Development state utilities:\n',state_table.round(3).to_string())
print('Recent short-window coverage:',audit.loc[('recent','recent_short'),'coverage'])
print('Historical errors (wrong old versions included):\n',audit.loc['historical'].round(3).to_string())
DEV_OUT.to_csv(OUT/'development_outcomes.csv',index=False)


### M2 · 策略解释与边界检查
短窗的近期题正确率高且便宜；长窗会增加成本，但历史查询未必落在窗口中。原始TF–IDF可能被词面相似的旧版本占满候选；resolved_TFIDF先消解its并统一同义词，仍然是词面检索，不是基于真值的oracle。

`coverage`只计“最后一条证据消息”入选；同值可能在更早消息重复出现，所以coverage与accuracy可以不同。所有候选先构造，耗时应合计，不能仅用最终所选臂构造时间声称节约了系统时间。

下面导出10条样例，包括新旧事实、focus、各策略所见消息和错误原因。`audit_examples.json`是机器可追溯全记录；`audit_compact.json`是适合逐条阅读的证据链；`human_audit_review.csv`留给组员本人填写，AI不代签。


In [ ]:
pool=[(ep,pr) for ep in DEV for pr in ep['probes']]
selected=[]
for qtype in ['recent','historical']:
    candidates=[(ep,pr) for ep,pr in pool if pr['qtype']==qtype]
    for old_flag,quota in [(True,3),(False,2)]:
        found=[x for x in candidates if bool(x[1]['old_values'])==old_flag][:quota]
        selected.extend(found)
    used={(ep['seed'],pr['t']) for ep,pr in selected}
    needed=5-sum(pr['qtype']==qtype for _,pr in selected)
    selected.extend([x for x in candidates if (x[0]['seed'],x[1]['t']) not in used][:needed])
assert len(selected)==10
examples=[]
for ep,pr in selected:
    view=public_view(ep,pr)
    contexts={a:select_context(view,a) for a in ARMS}
    examples.append({'seed':ep['seed'],'t':pr['t'],'public_view':view,
        'arms':{a:{'context':ctx,'answer':rule_reader(view['question'],ctx),
                   'cost':units(prompt(view['question'],ctx))} for a,ctx in contexts.items()},
        'evaluator_only':{k:pr[k] for k in ['gold','qtype','state','old_values','evidence']}})
(OUT/'audit_examples.json').write_text(json.dumps(examples,ensure_ascii=False,indent=2),encoding='utf-8')


In [ ]:
compact=[]
for e in examples:
    q=e['public_view']['question']; h=e['public_view']['history']
    tr=independent_trace(q,h)
    matching=[m for m in h if parse_fact(m['text']) and parse_fact(m['text'])[:2]==(tr['project'],tr['attribute'])]
    compact.append(dict(seed=e['seed'],t=e['t'],question=q,trace=tr,
        target_history=matching, arms={a:dict(answer=v['answer'],cost=v['cost'],
            message_ids=[m['id'] for m in v['context']],
            error='correct' if v['answer']==tr['gold'] else ('unknown' if v['answer']=='UNKNOWN' else 'old_value' if v['answer'] in tr['old_values'] else 'other'))
            for a,v in e['arms'].items()}))
save_json('audit_compact.json',compact)
AUDIT_HASH=hashlib.sha256(json.dumps(examples,sort_keys=True).encode()).hexdigest()
# 模板只写入当前OUT；永不自动改写团队人工记录。
PROTOCOL = dict(CFG, n_eval=30)
PROTOCOL.pop('eval_seed')  # 正式seed由全组在同一份审查记录里确认
PROTOCOL.update(primary='LaplaceCTS minus development-best-fixed per scenario',
                secondary='TS exploratory', analysis='paired dialogue bootstrap; 97.5% primary intervals')
PROTOCOL_HASH = hashlib.sha256(json.dumps(PROTOCOL,sort_keys=True).encode()).hexdigest()
REVIEW_TEMPLATE = dict(code_sha256=CODE_HASH, audit_sha256=AUDIT_HASH, protocol_sha256=PROTOCOL_HASH,
    formal_seed=None, seed_not_previously_used=False,
    reviews=[dict(seed=e['seed'],t=e['t'],m1_reviewer='',m2_reviewer='',verdict='pending',notes='',m2_notes='') for e in examples],
    approvals={m:dict(name='',approved=False,notes='',evidence=f'deliverables/{m}_notes.md') for m in ['M1','M2','M3','M4','M5']})
save_json('review_template.json',REVIEW_TEMPLATE)
save_json('protocol.json',PROTOCOL)
def validate_team_review(review):
    assert review.get('code_sha256')==CODE_HASH, '代码变化，需重新审查。'
    assert review.get('audit_sha256')==AUDIT_HASH, '10题证据变化，需重新审查。'
    assert review.get('protocol_sha256')==PROTOCOL_HASH, '实验口径变化，需重新审查。'
    rows=review.get('reviews',[])
    assert len(rows)==10 and {(r['seed'],r['t']) for r in rows}=={(e['seed'],e['t']) for e in examples}
    for r in rows:
        assert r['verdict']=='pass' and all(str(r.get(k,'')).strip() for k in ['m1_reviewer','m2_reviewer','notes','m2_notes']), '10题需双人署名并留下解释。'
        assert r['m1_reviewer'].strip()!=r['m2_reviewer'].strip(), 'M1/M2须由不同成员复核。'
    names=[]
    for member in ['M1','M2','M3','M4','M5']:
        approval=review.get('approvals',{}).get(member,{})
        assert approval.get('approved') is True and str(approval.get('name','')).strip() and str(approval.get('notes','')).strip(), f'{member}待签名确认。'
        evidence=(ROOT/approval['evidence']).resolve()
        assert evidence.is_relative_to(ROOT/'deliverables') and evidence.is_file(), f'{member}说明文件不存在。'
        assert '待填写' not in evidence.read_text(encoding='utf-8'), f'{member}说明仍有待填写项。'
        names.append(approval['name'].strip())
    assert len(set(names))==5, '五位成员分别署名。'
    assert all(r['m1_reviewer'].strip()==names[0] and r['m2_reviewer'].strip()==names[1] for r in rows), '逐题审查人与成员署名不一致。'
    assert type(review.get('formal_seed')) is int and review.get('seed_not_previously_used') is True, 'M4须确认seed未使用。'
    return True

review_valid=False
try:
    review_valid=validate_team_review(TEAM_REVIEW)
except (AssertionError, KeyError, TypeError, ValueError, OSError) as exc:
    print('人工审查尚未通过（demo可继续）：',exc)
CHECKS['team_review_valid']=review_valid
CHECKS['m2_development_arm_rows']=len(DEV_OUT)
assert len(DEV_OUT)==len(DEV)*CFG['T']*len(ARMS)
for a in ARMS: assert select_context({'question':'What is its current dataset?','history':[]},a)==[]
# 同一个公开前缀之后追加未来事实，不应影响该轮选择。
ep=DEV[0]; pr=ep['probes'][0]
future=dict(ep,history=ep['history']+[{'id':len(ep['history']),'text':'Focus is project P999.'}])
assert all(select_context(public_view(ep,pr),a)==select_context(public_view(future,pr),a) for a in ARMS)
CHECKS['m2_empty_history_future_isolation_budget_order']='passed'
audit.to_csv(OUT/'development_strategy_by_query.csv')
# 有目的的边界案例，不用这几题估计总体错误率；包含强检索失败。
boundary=[]
conditions=[('recent_tfidf_failure',(DEV_OUT.qtype=='recent') & (DEV_OUT.arm=='tfidf') & (DEV_OUT.y==0)),
            ('historical_tfidf_old',(DEV_OUT.qtype=='historical') & (DEV_OUT.arm=='tfidf') & (DEV_OUT.outdated==1)),
            ('resolved_old',(DEV_OUT.arm=='resolved_tfidf') & (DEV_OUT.outdated==1)),
            ('correct_without_latest_record',(DEV_OUT.y==1) & (DEV_OUT.coverage==0))]
for label,mask in conditions:
    candidates=DEV_OUT[mask]
    if candidates.empty: continue
    row=candidates.iloc[0]; ep=next(e for e in DEV if e['seed']==row.seed); pr=ep['probes'][int(row.t)]
    view=public_view(ep,pr); ctx=select_context(view,row.arm)
    boundary.append(dict(case=label,seed=int(row.seed),t=int(row.t),arm=row.arm,
        question=view['question'],gold=pr['gold'],answer=rule_reader(view['question'],ctx),
        latest_evidence=ep['history'][pr['evidence']],selected_context=ctx,
        latest_public_focus=next(m for m in reversed(view['history']) if FOCUS.search(m['text']))))
save_json('strategy_boundary_cases.json',boundary)
display(audit.round(4))
display(pd.DataFrame([dict(seed=e['seed'],t=e['t'],question=e['question'],
                           gold=e['trace']['gold'],**{a:v['error'] for a,v in e['arms'].items()}) for e in compact]))
print('Audit hash:',AUDIT_HASH,'| Team review valid:',review_valid)


<a id="member-m3"></a>
## M3 · 贝叶斯学习器与方法核查
**输入：** M2公开接口、DEV_OUT。**输出：** Selector/replay、method_checks.json、`deliverables/M3_notes.md`；M4接收并复核。

1. 先手算Beta(1,1)收到1、1、0后的分布和均值，再阅读TS代码；成本只在选臂时扣除，不写入失败次数。
2. 对照下一节推导logistic MAP梯度、Hessian及其逆；解释高斯先验、三个特征的共线性、每5次已选反馈重拟合。
3. 运行数值检查，逐项解释有限差分、协方差正定、对话重置、TS/DTS一致和未选奖励隔离分别检查什么。
4. 向M4演示 `choose(question, costs) -> arm_index` 和 `update(arm_index, y, question)`；全臂离线评估表只由回放器访问。
5. 在M3说明中写公式、检查数值、局限与约4分钟讲稿；在review.json签署M3 approval。自动检查通过不代表CTS必须胜出。


### 4 · M3：解释并核查两个贝叶斯学习器
**目标：** 把课程推导对应到代码，说明不确定性如何推动探索；不要求发明新算法。

**TS：** 每臂正确率 $p_a\sim\mathrm{Beta}(1,1)$；观测该臂S次正确、F次错误后为 $\mathrm{Beta}(1+S,1+F)$。每轮采样正确率并减已知成本选臂；仅更新所选臂。手算检查：答对2次、答错1次应得Beta(3,2)，后验均值0.6。

**LaplaceCTS：** $x=(1,\mathbf1[项目名],\mathbf1[its])$，每臂 $P(Y=1|x,w_a)=\sigma(x^T w_a)$，先验 $w_a\sim N(0,I)$。MAP最小化logistic负对数似然加 $\|w_a\|^2/2$；协方差近似为 $(I+X^TWX)^{-1}$。从近似后验采样参数，再减成本选臂。

每臂累计5次新反馈才重拟合一次，未到5次用上一后验（初始为先验）；每条新对话重新初始化学习器。开发集不用于预训练正式对话的参数。

**课程：** Lecture 2 pp.16–22、49–50；Lecture 6 pp.11–15、27–30。这是课程概念组合后的项目实现，不声称课件直接给出了完整LaplaceCTS算法。

**边界：** 当前模板中“项目名”和“its”互补，与截距线性相关；高斯先验使Hessian可逆，但不宜解释单个系数的独立效应。特征被设计为可识别题型。TS也知道各臂当前成本，“不看题”准确地说是其成功率模型不使用问题特征。

RuleRouter对指代题用短窗，其他题用强检索，是人工规则参考，不是理论上限。DTS只保留开发γ表；UCB/Markov在附录，不纳入主结论。


In [ ]:
class Selector:
    def __init__(self,name,seed,gamma=1.):
        assert name in ARMS+['TS','DTS','LaplaceCTS','RuleRouter','UCB'], name
        self.name=name; self.rng=np.random.default_rng(seed); self.gamma=gamma
        self.s=np.zeros(len(ARMS));self.f=self.s.copy();self.t=0
        self.X=[[] for _ in ARMS];self.Y=[[] for _ in ARMS]
        self.mu=np.zeros((len(ARMS),3));self.cov=np.tile(np.eye(3)/CFG['prior_precision'],(len(ARMS),1,1))
    def choose(self,q,cost):
        self.t+=1
        if self.name in ARMS:return ARMS.index(self.name)
        if self.name=='RuleRouter':return 0 if features(q)[2] and not features(q)[1] else 3
        if self.name=='LaplaceCTS':
            p=np.array([expit(features(q)@self.rng.multivariate_normal(self.mu[k],self.cov[k])) for k in range(len(ARMS))])
        elif self.name=='UCB':
            n=self.s+self.f
            if (n==0).any():return int(np.flatnonzero(n==0)[0])
            p=self.s/n+np.sqrt(2*np.log(self.t)/n)
        else:
            if self.name=='DTS': self.s*=self.gamma;self.f*=self.gamma
            p=self.rng.beta(1+self.s,1+self.f)
        return int(np.argmax(p-CFG['lam']*cost/CFG['cref']))
    def update(self,k,y,q):
        self.s[k]+=y;self.f[k]+=1-y
        if self.name!='LaplaceCTS':return
        self.X[k].append(features(q)); self.Y[k].append(y)
        if len(self.Y[k])%CFG['refit_every']:return
        X=np.array(self.X[k]);Y=np.array(self.Y[k])
        def fg(w):
            z=X@w; return np.logaddexp(0,z).sum()-Y@z+.5*CFG['prior_precision']*w@w, X.T@(expit(z)-Y)+CFG['prior_precision']*w
        fit=minimize(fg,self.mu[k],jac=True,method='BFGS',options={'gtol':1e-6,'maxiter':100})
        if not fit.success and np.linalg.norm(fit.jac)>1e-3:raise RuntimeError(fit.message)
        self.mu[k]=fit.x;p=expit(X@fit.x)
        self.cov[k]=np.linalg.inv(CFG['prior_precision']*np.eye(3)+X.T@((p*(1-p))[:,None]*X))

def replay(frame,name,alg_seed=17,gamma=1.):
    # 评估器拥有全臂反事实；choose仍只得到q/cost，update只得到选中y。
    selector=Selector(name,alg_seed,gamma); rows=[]
    expected=pd.MultiIndex.from_product([sorted(frame.t.unique()),ARMS],names=['t','arm'])
    g=frame.set_index(['t','arm'])
    assert g.index.is_unique and len(g)==len(expected)
    g=g.reindex(expected); assert not g.isna().any().any()
    y=g.y.to_numpy().reshape(-1,len(ARMS)); cost=g.cost.to_numpy().reshape(-1,len(ARMS))
    outdated=g.outdated.to_numpy().reshape(-1,len(ARMS))
    coverage=g.coverage.to_numpy().reshape(-1,len(ARMS))
    first=g.iloc[::len(ARMS)]
    for j,(t,q) in enumerate(zip(sorted(frame.t.unique()),first.question)):
        k=selector.choose(q,cost[j].copy())
        selected_y=int(y[j,k]); selector.update(k,selected_y,q)
        rows.append(dict(seed=int(first.seed.iloc[j]),scenario=first.scenario.iloc[j],t=int(t),method=name,
            alg_seed=alg_seed,arm=ARMS[k],y=selected_y,cost=int(cost[j,k]),
            utility=selected_y-CFG['lam']*cost[j,k]/CFG['cref'],outdated=int(outdated[j,k]),coverage=int(coverage[j,k])))
    return pd.DataFrame(rows)
# Core invariants: gamma=1 DTS exactly reproduces TS; only selected counts increase.
tiny=DEV_OUT[DEV_OUT.seed==DEV[0]['seed']]
a=replay(tiny,'TS');b=replay(tiny,'DTS',gamma=1.)
assert a.arm.tolist()==b.arm.tolist()
s=Selector('TS',0);s.update(2,1,'What is its dataset?'); assert np.array_equal(s.s,[0,0,1,0])
assert not any(x in public_view(DEV[0],DEV[0]['probes'][0]) for x in ['gold','state','qtype'])
print('TS/DTS identity, selected-only updates and public-view checks passed.')


In [ ]:
# M3：检查的是反馈可见性与数值后验，而非“方法一定胜出”。
q_check='What is its current dataset?'
s_check=Selector('LaplaceCTS',17)
for y_check in [1,1,0,1,0]: s_check.update(1,y_check,q_check)
assert [len(y) for y in s_check.Y]==[0,5,0,0]
assert np.allclose(s_check.mu[[0,2,3]],0)
assert np.linalg.eigvalsh(s_check.cov[1]).min()>0
assert np.allclose(s_check.cov[1],s_check.cov[1].T)
assert np.allclose(features(q_check),[1,0,1])
print('LaplaceCTS selected-only feedback, positive-definite covariance, feature checks passed.')


### M3 · 手算、推导与实现对应
Beta(1,1)连续观测1、1、0后为Beta(3,2)，均值3/5；成本惩罚不作为“失败次数”，只在选择时减去已知成本。

MAP目标为 $L(w)=\sum_i[\log(1+e^{x_i^Tw})-y_ix_i^Tw]+\frac12w^Tw$，梯度为 $X^T(\sigma(Xw)-y)+w$，Hessian为 $I+X^T\operatorname{diag}(p_i(1-p_i))X$。因为先验精度为1，即使三个特征共线，Hessian仍正定。这里反演**解析Hessian**，不把BFGS自带的近似逆矩阵当Laplace协方差。

每臂第五次反馈才重拟合，间隔内使用上一次后验，是计算开销折中。每条对话、每个算法种子重新实例化Selector。下格检查有限差分梯度/Hessian、MAP驻点、协方差、更新频率、Beta手算和未选奖励不影响更新；这比仅检查代码能运行更直接。

**局限：** Laplace为近似后验；TS和CTS模型族不同，差异不是纯特征因果消融；当前模板特征易于识别，无法据此外推真实对话；奖励即时且无噪声，不模拟LLM评价误差。


In [ ]:
beta=Selector('TS',0)
for y0 in [1,1,0]: beta.update(2,y0,'What is its current dataset?')
assert (1+beta.s[2],1+beta.f[2])==(3,2)
assert (1+beta.s[2])/(2+beta.s[2]+beta.f[2])==.6
assert np.array_equal(beta.s[[0,1,3]],np.zeros(3))
q0='What is its current dataset?'; q1='What is P1 current metric?'
fit_test=Selector('LaplaceCTS',0)
for yy,qq in zip([1,0,1,0],[q0,q1,q0,q1]): fit_test.update(1,yy,qq)
assert np.array_equal(fit_test.mu,np.zeros((4,3)))  # 尚未第5次
fit_test.update(1,1,q0)
X=np.array(fit_test.X[1]); Y=np.array(fit_test.Y[1]); w=fit_test.mu[1]
def objective(w): return np.logaddexp(0,X@w).sum()-Y@(X@w)+.5*w@w
def gradient(w): return X.T@(expit(X@w)-Y)+w
eps=1e-5; I=np.eye(3)
fd=np.array([(objective(w+eps*e)-objective(w-eps*e))/(2*eps) for e in I])
p=expit(X@w); H=I+X.T@((p*(1-p))[:,None]*X)
fdH=np.column_stack([(gradient(w+eps*e)-gradient(w-eps*e))/(2*eps) for e in I])
assert np.allclose(fd,gradient(w),atol=1e-7)
assert np.allclose(fdH,H,atol=1e-7) and np.linalg.norm(gradient(w))<1e-3
assert np.allclose(H@fit_test.cov[1],I,atol=1e-8)
assert np.linalg.eigvalsh(fit_test.cov[1]).min()>0
assert np.array_equal(Selector('LaplaceCTS',0).mu,np.zeros((4,3)))
# 优化后的数组回放必须与原逐轮DataFrame实现一致，包括随机数调用顺序。
def reference_replay_actions(frame,name,seed):
    model=Selector(name,seed); actions=[]
    for _,group in frame.groupby('t',sort=True):
        group=group.set_index('arm').loc[ARMS]; q=group.question.iloc[0]
        k=model.choose(q,group.cost.to_numpy()); model.update(k,int(group.iloc[k].y),q)
        actions.append(ARMS[k])
    return actions
for name in ['TS','LaplaceCTS','RuleRouter']:
    assert reference_replay_actions(tiny,name,17)==replay(tiny,name,17).arm.tolist()
CHECKS['m3_optimized_replay_matches_reference']='passed on full development dialogue'
# 逐轮改变未选反事实奖励：同样已选反馈下，动作及后验必须相同。
for name in ['TS','LaplaceCTS']:
    a=Selector(name,73); b=Selector(name,73)
    for t0 in range(40):
        qq=q0 if t0%2 else q1; cc=np.array([120,510,260,270])
        ka=a.choose(qq,cc); kb=b.choose(qq,cc); assert ka==kb
        all_y=np.array([t0%2,1,0,1]); changed=1-all_y; changed[ka]=all_y[ka]
        a.update(ka,int(all_y[ka]),qq); b.update(kb,int(changed[kb]),qq)
        assert np.array_equal(a.s,b.s) and np.allclose(a.mu,b.mu) and np.allclose(a.cov,b.cov)
CHECKS['m3_beta_hand_calculation']='Beta(3,2), mean=0.6'
CHECKS['m3_gradient_hessian_covariance_refit_reset_feedback']='passed'
save_json('method_checks.json',{'checks':CHECKS,'map_gradient_norm':float(np.linalg.norm(gradient(w))),
    'hessian_fd_max_error':float(np.max(np.abs(fdH-H))),
    'covariance_min_eigenvalue':float(np.linalg.eigvalsh(fit_test.cov[1]).min())})
print('M3: Beta hand calculation, finite differences, posterior covariance, reset and feedback isolation passed.')


## 人工交接点 · development命令到此结束
已生成的代码结果是待复核证据。M1/M2填写10题审计；M1–M5分别完成自己的说明与approval，M4填写尚未使用的正式seed并登记。此时M4写实验计划、M5写slides结构与接收标准即可，不能预写正式结果。

全组签名意味着确认**同一份代码、审计、实验口径及formal_seed**。保存review.json并提交Git；不要在签名后擅改seed或规则。脚本会检查字段、哈希与说明文件，内容真实性仍由签名者负责。代码改动后先重新执行development并复核新模板。


<a id="member-m4"></a>
## M4 · 冻结、正式实验与统计交接
**先决条件：** M1/M2逐题双签、M3方法核查、五位成员对口径和seed的确认。M4是唯一正式运行及共享notebook合并负责人。

1. 在demo查看开发表，解释每场景最佳固定臂；DTS只用于开发补充，不能事后加入主比较。
2. 检查 `team_workspace/review.json` 的真实签名和说明，确认正式seed未使用、不同于可行性版本20261105及其对话区间；运行前提交代码与审查记录。
3. 运行 `python scripts/run_notebook.py --stage m4 --mode formal`。程序在生成正式数据前核验审查，保存冻结配置/源码/证据副本，登记seed。formal目录不覆盖，登记过的seed不作为新的首次正式实验重跑。
4. 核对2场景×30条×300题，保留全部正负结果及失败目录；出错时在formal_runs.json记录原因和数据是否已看，不可改seed反复跑至显著。修正分析应另开明确标注的版本，先保留原始证据。
5. 按同一对话配对、先平均算法seed再bootstrap。主结论只看两场景校正97.5%区间；TS为探索；图表和阶段分析不扩充主假设。
6. 把**完整experiment目录**、manifest校验和、代码commit及主结论交M3复核，再交M5。填写M4说明的结果部分与进度证据路径。

开发默认每场景3条；demo每场景6条，仅验证流程。正式每场景30条，算法seed17/37不是独立样本。允许正式对话中按冻结程序在线更新，不从开发集迁移已拟合后验。


In [ ]:
BEST_FIXED=DEV_OUT.groupby(['scenario','arm']).utility.mean().unstack().idxmax(axis=1).to_dict()
tuning=[]
for gamma in CFG['gamma_grid']:
    vals=[replay(g,'DTS',s,gamma).utility.mean() for _,g in DEV_OUT.groupby('seed') for s in CFG['algorithm_seeds']]
    tuning.append({'gamma':gamma,'development_utility':float(np.mean(vals))})
GAMMA_TABLE=pd.DataFrame(tuning); BEST_GAMMA=float(GAMMA_TABLE.loc[GAMMA_TABLE.development_utility.idxmax(),'gamma'])
GAMMA_TABLE.to_csv(OUT/'dts_development_gamma.csv',index=False)
print('Fixed baselines, chosen ONLY on development:',BEST_FIXED)
print('DTS appendix table:\n',GAMMA_TABLE.round(4).to_string(index=False))
if MODE=='formal':
    # 从磁盘重新读取，防止交互内核缓存旧签名。
    TEAM_REVIEW=json.loads(review_file.read_text(encoding='utf-8'))
    review_valid=validate_team_review(TEAM_REVIEW)
    assert TEAM_REVIEW['formal_seed']==CFG['eval_seed'], '配置与已签seed不一致。'
    ledger_path=TEAM_DIR/'formal_runs.json'
    ledger=json.loads(ledger_path.read_text(encoding='utf-8'))
    proposed={CFG['eval_seed']+100*s+i for s in range(2) for i in range(CFG['n_eval'])}
    used={seed for run in ledger for seed in run['dialogue_seeds']}
    assert proposed.isdisjoint(used), '已登记/已查看seed不得作为新的首次正式实验；保留失败记录并说明修正。'
    save_json('team_review_snapshot.json',TEAM_REVIEW)
    for member,approval in TEAM_REVIEW['approvals'].items():
        shutil.copy2(ROOT/approval['evidence'],OUT/f'{member}_notes_at_freeze.md')
FROZEN=dict(version=CFG['version'],mode=MODE,config=CFG,scenarios=SCENARIOS,
    best_fixed_by_scenario=BEST_FIXED,primary=PROTOCOL['primary'],secondary=PROTOCOL['secondary'],
    analysis=PROTOCOL['analysis'],code_sha256=CODE_HASH,protocol_sha256=PROTOCOL_HASH,
    audit_sha256=AUDIT_HASH,review_valid=review_valid,
    review_sha256=digest_file(OUT/'team_review_snapshot.json') if MODE=='formal' else None,
    scope='controlled simulation; no real LLM; proxy units',
    frozen_at=datetime.now().astimezone().isoformat(),
    prior_evidence='Completed feasibility notebook seen; not new team formal evidence')
FREEZE_TEXT=json.dumps(FROZEN,sort_keys=True)
FREEZE_HASH=hashlib.sha256(FREEZE_TEXT.encode()).hexdigest()
(OUT/'frozen_config.json').write_text(FREEZE_TEXT,encoding='utf-8')
if MODE=='formal':
    ledger.append(dict(run=RUN_ROOT.relative_to(ROOT).as_posix(),seed=CFG['eval_seed'],
        dialogue_seeds=sorted(proposed),status='started',code_sha256=CODE_HASH,
        frozen_sha256=FREEZE_HASH,data_seen='unknown until M4 records',notes=''))
    temporary=ledger_path.with_suffix('.tmp')
    temporary.write_text(json.dumps(ledger,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    temporary.replace(ledger_path)
print('Frozen:',OUT/'frozen_config.json')


### M4 · 正式数据与逐轮回放
2场景×30条独立对话×300题=18,000题；四臂反事实评价72,000行。TS、CTS各用17/37两个随机种子，固定臂及RuleRouter为确定方法。先在同一对话内平均种子，再做对话级统计。**60条是独立数据规模，不是120条或18,000个独立样本。**

每条对话归档结果、核查全部真值并打印进度。算法只看到被选臂反馈；反事实表用于公平回放和审计。每条新对话都从先验重新开始。所有方法具有相同即时反馈假设。


In [ ]:
assert json.dumps(FROZEN,sort_keys=True)==FREEZE_TEXT
assert CFG==FROZEN['config']
EVAL=[generate(CFG['eval_seed']+100*s+i,sc) for s,sc in enumerate(SCENARIOS) for i in range(CFG['n_eval'])]
assert set(e['seed'] for e in DEV).isdisjoint(e['seed'] for e in EVAL)
assert len({e['seed'] for e in EVAL})==len(EVAL)
save_episodes(EVAL,'evaluation_dialogues.jsonl.gz')
METHODS=ARMS+['TS','LaplaceCTS','RuleRouter']
evaluated=[]; replayed=[]; truth_count=0
for i,e in enumerate(EVAL,1):
    truth_count+=audit_episode_truth(e)
    f=evaluate_episode(e); evaluated.append(f)
    f.to_csv(OUT/'checkpoints'/f"outcomes_{e['seed']}.csv",index=False)
    rr=pd.concat([replay(f,m,s) for m in METHODS
                  for s in (CFG['algorithm_seeds'] if m in ['TS','LaplaceCTS'] else [17])],ignore_index=True)
    rr.to_csv(OUT/'checkpoints'/f"replay_{e['seed']}.csv",index=False); replayed.append(rr)
    print(f"{MODE} {i}/{len(EVAL)} | {e['scenario']} seed={e['seed']} | elapsed={time.perf_counter()-START:.1f}s",flush=True)
EVAL_OUT=pd.concat(evaluated,ignore_index=True); RESULTS=pd.concat(replayed,ignore_index=True)
assert len(EVAL_OUT)==len(EVAL)*CFG['T']*len(ARMS)
assert EVAL_OUT.groupby(['scenario','seed','t']).size().eq(len(ARMS)).all()
assert EVAL_OUT.groupby('scenario').seed.nunique().eq(CFG['n_eval']).all()
assert not RESULTS.duplicated(['seed','t','method','alg_seed']).any()
assert RESULTS.groupby(['seed','method','alg_seed']).size().eq(CFG['T']).all()
assert len(RESULTS)==len(EVAL)*CFG['T']*(len(ARMS)+1+2*len(CFG['algorithm_seeds']))
assert np.isfinite(RESULTS[['y','cost','utility']]).all().all()
EVAL_OUT.to_csv(OUT/'evaluation_outcomes.csv',index=False)
CHECKS['m4_independent_test_truth_questions']=truth_count
CHECKS['m4_balanced_grid_replay_seed_separation']='passed'
print('Evaluation complete:',len(EVAL),'dialogues;',len(EVAL_OUT),'arm outcomes;',len(RESULTS),'replay rows')


### 7 · M4：按独立对话统计，解释效应而非只看显著性
**目标：** 对同一对话中的方法与固定基线做配对比较，避免把相关的300题当成300个独立样本。

每条对话先平均算法种子与题目，得到效用差；再在同场景内按整条对话重采样2000次。主表报告LaplaceCTS与固定基线的效用差、95%区间和两场景Bonferroni校正的97.5%区间（bootstrap近似覆盖，非有限样本保证）。主比较用97.5%区间判断正收益；TS为探索性。

点估计≥0.02是预先指定的实际收益参考，并不表示“置信地超过0.02”；若要这样说，需要区间下界也超过0.02。跨0表示证据不足，不能直接说方法等效。TS与LaplaceCTS的模型结构也不同，二者差异不能作为问题特征的严格因果消融。

**验收：** primary表每场景一行，formal中n=30；保留负结果。读出效应、区间、适用场景与局限，交M5。不要把累计效用差命名为相对最优策略的理论regret。


In [ ]:
EP=RESULTS.groupby(['scenario','seed','method'])[['y','cost','utility','outdated','coverage']].mean().reset_index()
def paired_ci(ep,method,baseline,scenario,nboot=CFG['nboot']):
    z=ep[ep.scenario==scenario].pivot(index='seed',columns='method',values='utility')
    assert len(z)==CFG['n_eval'] and not z[[method,baseline]].isna().any().any()
    d=(z[method]-z[baseline]).to_numpy();rng=np.random.default_rng(CFG['bootstrap_seed'])
    boot=d[rng.integers(0,len(d),size=(nboot,len(d)))].mean(axis=1)
    return dict(scenario=scenario,method=method,baseline=baseline,n=len(d),effect=d.mean(),
                low=np.quantile(boot,.025),high=np.quantile(boot,.975),
                family_low=np.quantile(boot,.0125),family_high=np.quantile(boot,.9875))
primary=pd.DataFrame([paired_ci(EP,'LaplaceCTS',BEST_FIXED[s],s) for s in SCENARIOS])
secondary=pd.DataFrame([paired_ci(EP,'TS',BEST_FIXED[s],s) for s in SCENARIOS])
primary['positive_family_interval']=primary.family_low>0
primary['effect_at_least_002']=primary.effect>=.02
print('PRIMARY —',MODE,'(formal inference only after frozen formal run):\n',primary.round(4).to_string(index=False))
print('SECONDARY — exploratory:\n',secondary.round(4).to_string(index=False))
print('RuleRouter is a reference, NOT an upper bound.')
primary.to_csv(OUT/'primary_effects.csv',index=False);secondary.to_csv(OUT/'secondary_effects.csv',index=False)
EP.to_csv(OUT/'episode_metrics.csv',index=False);RESULTS.to_csv(OUT/'replay.csv',index=False)
# Before/after descriptive table for handoff, no claim of exact adaptation time.
z=RESULTS[RESULTS.scenario=='abrupt'].copy();z['phase']=np.where(z.t<CFG['T']//2,'before','after')
phase=z.groupby(['method','phase'])[['utility','y','cost']].mean()
phase.to_csv(OUT/'abrupt_before_after.csv');print(phase.loc[['TS','LaplaceCTS','RuleRouter',BEST_FIXED['abrupt']]].round(3).to_string())


In [ ]:
construction=EVAL_OUT.groupby(['scenario','arm']).construction_ms.agg(['mean','median','max'])
construction.to_csv(OUT/'candidate_construction_ms.csv')
display(construction.round(3))
print('构造时间受硬件与缓存影响；这是各候选的构造耗时，不能推断真实LLM加速。')


In [ ]:
summary=EP.groupby(['scenario','method'])[['y','cost','utility','outdated','coverage']].mean()
summary.to_csv(OUT/'method_summary.csv'); display(summary.round(4))
selection=RESULTS.merge(EVAL_OUT[['seed','t','qtype','state']].drop_duplicates(),on=['seed','t'],validate='many_to_one')
shares=selection.groupby(['scenario','method','qtype']).arm.value_counts(normalize=True).rename('share').reset_index()
shares.to_csv(OUT/'selection_by_question.csv',index=False)
diffs=[]
for scenario in SCENARIOS:
    wide=EP[EP.scenario==scenario].pivot(index='seed',columns='method',values='utility')
    for seed,row in wide.iterrows():
        diffs.append(dict(scenario=scenario,seed=int(seed),baseline=BEST_FIXED[scenario],
            cts_minus_fixed=row['LaplaceCTS']-row[BEST_FIXED[scenario]],
            ts_minus_fixed=row['TS']-row[BEST_FIXED[scenario]],
            cts_minus_router=row['LaplaceCTS']-row['RuleRouter']))
pd.DataFrame(diffs).to_csv(OUT/'paired_dialogue_differences.csv',index=False)
# 同一轨迹按每个算法种子分别核对；仅描述随机性，不能当独立对话。
per_rng=RESULTS.groupby(['scenario','seed','method','alg_seed']).utility.mean().reset_index()
per_rng.to_csv(OUT/'algorithm_seed_metrics.csv',index=False)
assert len(EP)==len(EVAL)*len(METHODS)
assert len(primary)==2 and primary.n.eq(CFG['n_eval']).all()
assert primary[['effect','low','high','family_low','family_high']].apply(np.isfinite).all().all()
# 配对bootstrap自身的必要性质：自身比较的所有效应与区间均为零。
zero=paired_ci(EP,ARMS[0],ARMS[0],SCENARIOS[0])
assert all(zero[k]==0 for k in ['effect','low','high','family_low','family_high'])
# 显式两阶段平均与groupby简写一致，避免把算法种子当样本。
explicit=RESULTS.groupby(['scenario','seed','method','alg_seed']).utility.mean().groupby(['scenario','seed','method']).mean()
assert np.allclose(explicit,EP.set_index(['scenario','seed','method']).utility)
CHECKS['m4_paired_bootstrap_and_seed_aggregation']='passed'
lines=['# M4计算交接：'+MODE, '',
    f"规模：{len(EVAL)}条独立对话，每场景n={CFG['n_eval']}。",'demo不用于正式推断。' if MODE=='demo' else '正式结果需M3交叉复核；包括不支持假设的结果。','']
for r in primary.itertuples():
    verdict='支持正收益' if r.family_low>0 else ('支持负收益' if r.family_high<0 else '证据不足，不能断言等效')
    lines.append(f'- {r.scenario} vs {r.baseline}: ΔU={r.effect:+.5f}; 97.5% CI [{r.family_low:+.5f}, {r.family_high:+.5f}]；{verdict}。点估计≥.02: {r.effect>=.02}；下界≥.02: {r.family_low>=.02}。')
lines+=['','M5只读取本目录manifest校验通过的CSV。成员任务与汇报仍由本人完成。',
    '解释边界：规则回答器、合成题型、即时无噪声反馈、proxy成本；RuleRouter不是理论上界。']
(OUT/'M4_handoff.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
save_json('validation_checks.json',CHECKS)
required=['frozen_config.json','notebook_snapshot.ipynb','development_dialogues.jsonl.gz',
    'evaluation_dialogues.jsonl.gz','development_outcomes.csv','evaluation_outcomes.csv','replay.csv',
    'episode_metrics.csv','primary_effects.csv','secondary_effects.csv','audit_examples.json',
    'audit_compact.json','review_template.json','method_checks.json','method_summary.csv',
    'paired_dialogue_differences.csv','M4_handoff.md','validation_checks.json']
assert all((OUT/p).is_file() for p in required)
current=json.loads(NOTEBOOK_PATH.read_text(encoding='utf-8'))
current_code='\n\n'.join(''.join(c['source']) for c in current['cells'] if c['cell_type']=='code')
assert hashlib.sha256(current_code.encode()).hexdigest()==CODE_HASH, '运行中源码已改变，请保留目录并排查。'
manifest=dict(status='complete',version=CFG['version'],mode=MODE,code_sha256=CODE_HASH,
    frozen_sha256=FREEZE_HASH,config=CFG,output_path=OUT.relative_to(ROOT).as_posix(),
    python=platform.python_version(),platform=platform.platform(),
    versions={m.__name__:m.__version__ for m in [np,pd,scipy,sklearn,matplotlib]},
    runtime_seconds=time.perf_counter()-START,dialogues=len(EVAL),questions=truth_count,
    arm_rows=len(EVAL_OUT),replay_rows=len(RESULTS),team_review_valid=review_valid,
    checks=CHECKS,files={p.relative_to(OUT).as_posix():digest_file(p) for p in sorted(OUT.rglob('*')) if p.is_file()})
save_json('manifest.json',manifest)
if MODE=='formal':
    ledger=json.loads(ledger_path.read_text(encoding='utf-8'))
    for run in ledger:
        if run['run']==RUN_ROOT.relative_to(ROOT).as_posix():
            run.update(status='complete',manifest_sha256=digest_file(OUT/'manifest.json'))
    temporary=ledger_path.with_suffix('.tmp')
    temporary.write_text(json.dumps(ledger,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    temporary.replace(ledger_path)
display(Markdown((OUT/'M4_handoff.md').read_text(encoding='utf-8')))
print('M4 COMPLETE:',OUT)


<a id="member-m5"></a>
## M5 · 七图、逐图解读与最终汇报
**输入：** M4交付的完整experiment目录与校验通过的manifest。**输出：** 新visualizations目录、M5说明、slides和视频。无需重跑M1–M4。

1. 第一天完成 `deliverables/M5_notes.md` 的slides骨架、成员发言安排和资料接收标准；正式冻结前的M5签名只确认计划，不声称结果已完成。
2. M4完成后，在新内核中设置 `ARIN_SOURCE` 为明确的experiment路径，再执行本节全部单元；或运行：
   `python scripts/run_notebook.py --stage m5 --source outputs/formal/seed_<seed>/experiment`
3. 先检查manifest、mode和每场景对话数。禁止自动选“最新目录”；demo图明确标注DEMO，不作为正式结果。
4. 七图逐张请责任人解释：M4/M3主效应与区间；M2全方法与收益分解；M3选择行为；M4累计差与阶段分析；M1/M2题型策略热图。M1另提供可追踪的对话例子用于背景页。
5. 根据本次CSV填写每图“观察→证据→局限”，保留TS和RuleRouter等全部对照，不照抄可行性本结论；更新 `deliverables/M5_notes.md`，约15–16页slides、15–20分钟视频按课程原始要求再核对。
6. M4校验图表来源，五人试讲确认姓名与贡献、图表数值、课程引用和AI辅助说明，最后使用 `deliverables/SUBMISSION_CHECKLIST.md` 交付。

本节图表自动生成中性标题与实际数值，不预设正收益。图2/5/7区间是描述性，累计带为逐点区间。图表输出：`M5_DIR/*.png`、`*.svg`、分析CSV、`M5_results_evaluation.md`、`manifest_m5.json`。


In [ ]:
# M5只读取明确指定、校验通过的M4归档。
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json, hashlib, time, textwrap
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display, Image, Markdown
import os
M5_STARTED=time.perf_counter()
if os.environ.get('ARIN_SOURCE'):
    M5_SOURCE=Path(os.environ['ARIN_SOURCE']).resolve()
elif 'OUT' in globals():
    M5_SOURCE=Path(OUT)
else:
    raise RuntimeError('请显式设置ARIN_SOURCE或用--stage m5 --source指定M4归档。')
assert M5_SOURCE.is_dir(), '请先运行M1–M4，或提供已有正式归档目录。'
def m5_sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
M5_MANIFEST=json.loads((M5_SOURCE/'manifest.json').read_text(encoding='utf-8'))
assert M5_MANIFEST['status']=='complete'
# 读取前校验整个归档，防止手动修改CSV后仍使用旧结论。
for name,sha in M5_MANIFEST['files'].items():
    assert (M5_SOURCE/name).resolve().is_relative_to(M5_SOURCE.resolve()), '非法归档路径'
    assert m5_sha(M5_SOURCE/name)==sha, f'归档文件发生变更：{name}'
# 核对实际执行的M5源码；交互修改后不能冒用旧快照的代码哈希。
if os.environ.get('ARIN_EXECUTION_NOTEBOOK'):
    M5_CODE_PATH=Path(os.environ['ARIN_EXECUTION_NOTEBOOK'])
else:
    M5_CODE_ROOT=next((p for p in [Path.cwd(), *Path.cwd().parents]
                       if (p/'ARIN7101_Context_Bandit_Release_Notebook.ipynb').is_file()), None)
    assert M5_CODE_ROOT is not None, '请从仓库启动交互内核。'
    M5_CODE_PATH=M5_CODE_ROOT/'ARIN7101_Context_Bandit_Release_Notebook.ipynb'
def m5_code_text(document):
    return '\n\n'.join(''.join(c['source']) for c in document['cells']
                       if c['cell_type']=='code' and 'm5' in c.get('metadata',{}).get('tags',[]))
M5_ARCHIVED_NOTEBOOK=json.loads((M5_SOURCE/'notebook_snapshot.ipynb').read_text(encoding='utf-8'))
assert m5_code_text(json.loads(M5_CODE_PATH.read_text(encoding='utf-8')))==m5_code_text(M5_ARCHIVED_NOTEBOOK), 'M5源码与归档不同；请使用--stage m5 --source运行归档快照，或明确建立修订版本。'
M5_CFG=M5_MANIFEST['config']; M5_MODE=M5_MANIFEST['mode']; M5_N=M5_CFG['n_eval']
M5_FROZEN=json.loads((M5_SOURCE/'frozen_config.json').read_text(encoding='utf-8'))
assert m5_sha(M5_SOURCE/'frozen_config.json')==M5_MANIFEST['frozen_sha256']
M5_BEST=M5_FROZEN['best_fixed_by_scenario']
M5_SCENARIOS=['stable','abrupt']; M5_ARMS=list(M5_CFG['budgets'])
M5_METHODS=['LaplaceCTS','RuleRouter','TS','recent_short','resolved_tfidf','recent_long','tfidf']
M5_NAMES={'recent_short':'Short window','recent_long':'Long window','tfidf':'TF-IDF',
          'resolved_tfidf':'Resolved TF-IDF','TS':'TS','LaplaceCTS':'LaplaceCTS','RuleRouter':'RuleRouter'}
M5_COLORS={'recent_short':'#267CA4','recent_long':'#A6B1BF','tfidf':'#CC9B59',
           'resolved_tfidf':'#27876C','TS':'#CE8B28','LaplaceCTS':'#B74E78','RuleRouter':'#455A64'}
M5_EP=pd.read_csv(M5_SOURCE/'episode_metrics.csv')
M5_REPLAY=pd.read_csv(M5_SOURCE/'replay.csv')
M5_OUTCOMES=pd.read_csv(M5_SOURCE/'evaluation_outcomes.csv')
M5_PRIMARY=pd.read_csv(M5_SOURCE/'primary_effects.csv')
M5_SECONDARY=pd.read_csv(M5_SOURCE/'secondary_effects.csv')
M5_SUMMARY=M5_EP.groupby(['scenario','method'])[['y','cost','utility','outdated','coverage']].mean()
M5_PHASE=pd.read_csv(M5_SOURCE/'abrupt_before_after.csv').set_index(['method','phase'])
M5_SHARES=pd.read_csv(M5_SOURCE/'selection_by_question.csv')
assert M5_EP.groupby('scenario').seed.nunique().eq(M5_N).all()
assert len(M5_PRIMARY)==2 and M5_PRIMARY.n.eq(M5_N).all()
M5_STAMP=datetime.now().strftime('%Y%m%d_%H%M%S_%f')
M5_DIR=M5_SOURCE.parent/'visualizations'/M5_STAMP
M5_DIR.mkdir(parents=True,exist_ok=False)
M5_FIGURES=[]
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10.5,'axes.titlesize':13,
    'axes.labelsize':11,'axes.spines.top':False,'axes.spines.right':False,
    'axes.facecolor':'#FAFBFC','figure.facecolor':'white','axes.edgecolor':'#A1ABB5',
    'grid.color':'#DDE3E8','grid.alpha':.7,'svg.fonttype':'none','figure.dpi':110})
M5_FOOT=f'{M5_MODE.upper()} | n={M5_N} independent dialogues/scenario | rule reader | proxy input cost'
def m5_save(fig,name,note):
    fig.text(.5,.018,M5_FOOT+'\n'+note,ha='center',va='bottom',fontsize=8.5,color='#53616E')
    fig.savefig(M5_DIR/(name+'.png'),dpi=170,bbox_inches='tight',facecolor='white')
    fig.savefig(M5_DIR/(name+'.svg'),bbox_inches='tight',facecolor='white')
    display(Image(filename=str(M5_DIR/(name+'.png'))))
    plt.close(fig);M5_FIGURES.append(name)
def m5_boot_mean(values,seed=2026):
    a=np.asarray(values);rng=np.random.default_rng(seed)
    means=a[rng.integers(0,len(a),size=(2000,len(a)))].mean(axis=1)
    return np.quantile(means,[.025,.975],axis=0)
print('Verified source:',M5_SOURCE.resolve())
print('M5 output:',M5_DIR.resolve())
display(M5_PRIMARY[['scenario','baseline','n','effect','family_low','family_high']].round(5))


### 图1 · 主比较与不确定性
粗95%/细97.5%区间区分开；0.02是效用差，不是准确率百分点。M4填写：两场景是否支持正收益？

把解读写入 `deliverables/M5_notes.md` 并注明本次图表路径；数据未产生前不填结论。


In [ ]:
fig,(ax,tab)=plt.subplots(1,2,figsize=(12,5.1),gridspec_kw={'width_ratios':[1.65,1]})
fig.subplots_adjust(left=.17,right=.98,top=.82,bottom=.21,wspace=.07)
labels=[]; rows=[]
for sc in M5_SCENARIOS:
    for method,table in [('LaplaceCTS',M5_PRIMARY),('TS',M5_SECONDARY)]:
        r=table[table.scenario==sc].iloc[0]; y=len(rows);color=M5_COLORS[method]
        if method=='LaplaceCTS':
            ax.plot([r.family_low,r.family_high],[y,y],color=color,lw=2)
            lo,hi=r.family_low,r.family_high
        else:lo,hi=r.low,r.high
        ax.plot([r.low,r.high],[y,y],color=color,lw=6,solid_capstyle='round')
        ax.scatter(r.effect,y,s=50,color=color,edgecolors='white',zorder=3)
        labels.append(f'{sc.title()} / {method}\nvs {M5_NAMES[r.baseline]}')
        rows.append((r.effect,lo,hi))
ax.axvline(0,color='#485563',ls='--',lw=1)
ax.axvline(.02,color='#27876C',ls=':',lw=1.4)
ax.set(yticks=range(4),yticklabels=labels,ylim=(3.65,-.6),
       xlabel='Mean utility difference per question')
ax.grid(axis='x');tab.set(ylim=(3.65,-.6),xlim=(0,1));tab.axis('off')
tab.text(.02,-.85,'Effect [CTS: 97.5%; TS: 95%]',weight='bold',fontsize=10)
for y,(effect,lo,hi) in enumerate(rows):tab.text(.02,y,f'{effect:+.4f}  [{lo:+.4f}, {hi:+.4f}]',va='center',fontsize=10.5)
fig.suptitle('01  |  Paired effects versus the development-selected fixed baseline',fontsize=14,y=.96)
m5_save(fig,'01_paired_effects','CTS: thick 95%, thin 97.5% paired bootstrap CI; TS: exploratory 95%. Dotted line: +0.02.')


### 图2 · 完整对照
同时比较四固定臂、TS、CTS、RuleRouter的效用/准确率/成本。M2填写：高准确率是否被成本抵消？

把解读写入 `deliverables/M5_notes.md` 并注明本次图表路径；数据未产生前不填结论。


In [ ]:
fig,axes=plt.subplots(2,3,figsize=(14,8.2),sharey=True)
fig.subplots_adjust(left=.14,right=.97,top=.85,bottom=.15,wspace=.25,hspace=.3)
yy=np.arange(len(M5_METHODS));colors=[M5_COLORS[m] for m in M5_METHODS]
for row,sc in enumerate(M5_SCENARIOS):
    s=M5_SUMMARY.loc[sc].reindex(M5_METHODS)
    for col,(metric,label,lim) in enumerate([('utility','Mean utility',(-.48,1.02)),('y','Accuracy (%)',(0,115)),('cost','Mean input cost (proxy units)',(0,650))]):
        ax=axes[row,col];values=s[metric].to_numpy()*(100 if metric=='y' else 1)
        ax.barh(yy,values,color=colors,height=.64,alpha=.93)
        if metric=='utility':
            ci=np.array([m5_boot_mean(M5_EP[(M5_EP.scenario==sc)&(M5_EP.method==m)].utility) for m in M5_METHODS])
            for j,(lo,hi) in enumerate(ci):ax.plot([lo,hi],[j,j],color='#182733',lw=1.2)
        for j,v in enumerate(values):
            offset=.012 if metric=='utility' else (1.4 if metric=='y' else 7)
            position=(max(v,ci[j,1])+.018 if v>=0 else min(v,ci[j,0])-.018) if metric=='utility' else (v+offset if v>=0 else v-offset)
            ax.text(position,j,f'{v:.3f}' if metric=='utility' else f'{v:.1f}',
                    ha='left' if v>=0 else 'right',va='center',fontsize=9)
        ax.axvline(0,color='#82919D',lw=.8);ax.grid(axis='x');ax.set_axisbelow(True)
        ax.set(xlim=lim,yticks=yy,yticklabels=[M5_NAMES[m] for m in M5_METHODS],xlabel=label,
               title=sc.title() if col==0 else '')
axes[0,0].invert_yaxis()
fig.suptitle('02  |  A complete comparison: utility, accuracy and input cost',fontsize=15,y=.96)
m5_save(fig,'02_all_method_metrics','Utility error bars: descriptive 95% dialogue bootstrap CI. Accuracy and cost: means; no additional primary tests.')


### 图3 · 收益来源分解
ΔU = ΔAccuracy − λΔCost/Cref。M2填写：本次两场景分别由哪项推动？

把解读写入 `deliverables/M5_notes.md` 并注明本次图表路径；数据未产生前不填结论。


In [ ]:
M5_DECOMP=[]
fig,axes=plt.subplots(1,2,figsize=(11.8,5.2),sharey=True)
fig.subplots_adjust(left=.09,right=.98,top=.79,bottom=.24,wspace=.17)
for ax,sc in zip(axes,M5_SCENARIOS):
    c=M5_SUMMARY.loc[(sc,'LaplaceCTS')];b=M5_SUMMARY.loc[(sc,M5_BEST[sc])]
    da=float(c.y-b.y);dc=float(-M5_CFG['lam']*(c.cost-b.cost)/M5_CFG['cref']);du=float(c.utility-b.utility)
    assert np.isclose(da+dc,du)
    M5_DECOMP.append(dict(scenario=sc,baseline=M5_BEST[sc],accuracy_delta=da,
                         cost_delta=float(c.cost-b.cost),cost_contribution=dc,utility_delta=du))
    ax.bar(0,da,color='#267CA4',width=.58)
    ax.bar(1,dc,bottom=da,color='#D59B3A',width=.58)
    ax.bar(2,du,color=M5_COLORS['LaplaceCTS'],width=.58)
    ax.plot([.29,.71],[da,da],color='#72828E',ls='--',lw=1)
    ax.plot([1.29,1.71],[du,du],color='#72828E',ls='--',lw=1)
    for j,value,endpoint in [(0,da,da),(1,dc,du),(2,du,du)]:
        ax.text(j,endpoint+(.004 if value>=0 else -.006),f'{value:+.4f}',ha='center',
                va='bottom' if value>=0 else 'top',weight='bold')
    ax.axhline(0,color='#71808A',lw=1);ax.grid(axis='y');ax.set_axisbelow(True)
    ax.set(xticks=[0,1,2],xticklabels=['Accuracy\ncontribution','Cost penalty\ncontribution','Net utility\nchange'],
           title=f'{sc.title()} vs {M5_NAMES[M5_BEST[sc]]}')
bounds=[0.]+[r[k] for r in M5_DECOMP for k in ['accuracy_delta','utility_delta']]
padding=max(.015,(max(bounds)-min(bounds))*.18)
axes[0].set_ylim(min(bounds)-padding,max(bounds)+padding)
axes[0].set_ylabel('Contribution to mean utility difference')
fig.suptitle('03  |  Accuracy and cost contributions to the observed utility difference',fontsize=14,y=.96)
pd.DataFrame(M5_DECOMP).to_csv(M5_DIR/'utility_decomposition.csv',index=False)
m5_save(fig,'03_utility_decomposition','Exact decomposition of observed mean differences; cost contribution = -0.5 x cost difference / 512.')


### 图4 · 随时间的选臂
每15题汇总动作比例，只在abrupt画需求切换线。M3填写：选臂变化与题型变化如何对应？

把解读写入 `deliverables/M5_notes.md` 并注明本次图表路径；数据未产生前不填结论。


In [ ]:
fig,axes=plt.subplots(2,2,figsize=(12.8,7.7),sharex=True,sharey=True)
fig.subplots_adjust(left=.08,right=.98,top=.79,bottom=.17,hspace=.3,wspace=.13)
M5_SELECTION_BINS=[]
for row,sc in enumerate(M5_SCENARIOS):
    for col,method in enumerate(['TS','LaplaceCTS']):
        ax=axes[row,col];z=M5_REPLAY[(M5_REPLAY.scenario==sc)&(M5_REPLAY.method==method)].copy()
        z['bin']=z.t//15
        shares=pd.crosstab(z.bin,z.arm,normalize='index').reindex(columns=M5_ARMS,fill_value=0)
        assert np.allclose(shares.sum(axis=1),1)
        exported=shares.reset_index();exported['scenario']=sc;exported['method']=method;M5_SELECTION_BINS.append(exported)
        xx=np.r_[shares.index.to_numpy()*15,M5_CFG['T']]
        ax.stackplot(xx,*[np.r_[shares[a],shares[a].iloc[-1]] for a in M5_ARMS],step='post',
                     colors=[M5_COLORS[a] for a in M5_ARMS],alpha=.95)
        if sc=='abrupt':ax.axvline(M5_CFG['T']//2,color='#142631',ls='--',lw=1.5)
        ax.set(title=f'{sc.title()} / {method}',ylim=(0,1),xlim=(0,M5_CFG['T']))
        if col==0:ax.set_ylabel('Selection share')
        if row==1:ax.set_xlabel('Question index (0-based; 15-question bins)')
fig.legend(handles=[Patch(facecolor=M5_COLORS[a],label=M5_NAMES[a]) for a in M5_ARMS],
           loc='upper center',bbox_to_anchor=(.5,.89),ncol=4,frameon=False)
fig.suptitle('04  |  Action selection over time and question mix',fontsize=15,y=.98)
pd.concat(M5_SELECTION_BINS).to_csv(M5_DIR/'selection_15_question_bins.csv',index=False)
m5_save(fig,'04_selection_over_time','Descriptive pooled choice proportions. Dashed line only in abrupt: question mix switches at index 150.')


### 图5 · 累计效用差
相对开发预选固定基线；逐点95%区间不是同时置信带，也不是理论regret。M4填写：切换后斜率如何？

把解读写入 `deliverables/M5_notes.md` 并注明本次图表路径；数据未产生前不填结论。


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12.8,5.5),sharey=True)
fig.subplots_adjust(left=.08,right=.98,top=.8,bottom=.22,wspace=.15)
M5_CURVES=[]
for ax,sc in zip(axes,M5_SCENARIOS):
    z=M5_REPLAY[M5_REPLAY.scenario==sc].groupby(['seed','t','method']).utility.mean().unstack('method')
    for method in ['TS','LaplaceCTS','RuleRouter']:
        per=(z[method]-z[M5_BEST[sc]]).unstack('t').sort_index(axis=1).cumsum(axis=1)
        assert per.shape==(M5_N,M5_CFG['T'])
        values=per.to_numpy();mean=values.mean(axis=0)
        # 与整条对话重采样等价的计数权重；不重采样单题。
        rng=np.random.default_rng(2026)
        weights=rng.multinomial(M5_N,np.full(M5_N,1/M5_N),size=2000)/M5_N
        lo,hi=np.quantile(weights@values,[.025,.975],axis=0)
        xx=np.arange(1,M5_CFG['T']+1)
        ax.fill_between(xx,lo,hi,color=M5_COLORS[method],alpha=.13)
        ax.plot(xx,mean,label=method,color=M5_COLORS[method],lw=2.2,ls='--' if method=='RuleRouter' else '-')
        for t,m,l,h in zip(xx,mean,lo,hi):M5_CURVES.append(dict(scenario=sc,method=method,questions_completed=int(t),mean=m,low=l,high=h))
        if method=='LaplaceCTS':assert np.isclose(mean[-1]/M5_CFG['T'],M5_PRIMARY.set_index('scenario').loc[sc,'effect'])
    ax.axhline(0,color='#72808A',lw=1)
    if sc=='abrupt':ax.axvline(M5_CFG['T']//2,color='#72808A',ls=':',lw=1.3)
    ax.set(title=f'{sc.title()} vs {M5_NAMES[M5_BEST[sc]]}',xlabel='Questions completed',xlim=(1,M5_CFG['T']))
    ax.grid(alpha=.45)
axes[0].set_ylabel('Mean cumulative utility difference per dialogue')
fig.legend(*axes[0].get_legend_handles_labels(),loc='upper center',bbox_to_anchor=(.5,.9),ncol=3,frameon=False)
fig.suptitle('05  |  Cumulative utility difference relative to a preselected fixed baseline',fontsize=15,y=.98)
pd.DataFrame(M5_CURVES).to_csv(M5_DIR/'cumulative_difference_bands.csv',index=False)
m5_save(fig,'05_cumulative_utility','Shading: pointwise 95% paired dialogue bootstrap intervals, not simultaneous bands or theoretical regret bounds.')


### 图6 · 题型与策略互补
全臂反事实只供评估器；在线学习器只看所选反馈。M1/M2填写：热图支持什么机制解释？

把解读写入 `deliverables/M5_notes.md` 并注明本次图表路径；数据未产生前不填结论。


In [ ]:
M5_ARM_QUERY=M5_OUTCOMES.groupby(['scenario','arm','qtype'])[['y','outdated','coverage','cost']].mean()
M5_ARM_QUERY.to_csv(M5_DIR/'arm_by_question.csv')
fig,axes=plt.subplots(1,2,figsize=(10.5,5),sharey=True)
fig.subplots_adjust(left=.19,right=.88,top=.79,bottom=.22,wspace=.2)
for ax,sc in zip(axes,M5_SCENARIOS):
    z=M5_ARM_QUERY.loc[sc].y.unstack('qtype').reindex(index=M5_ARMS,columns=['recent','historical'])
    im=ax.imshow(z,vmin=0,vmax=1,cmap='Blues',aspect='auto')
    for i in range(4):
        for j in range(2):ax.text(j,i,f'{z.iloc[i,j]:.1%}',ha='center',va='center',fontsize=13,
                                  color='white' if z.iloc[i,j]>.55 else '#18364B',weight='bold')
    ax.set(xticks=[0,1],xticklabels=['Recent / its','Historical / explicit'],yticks=range(4),
           yticklabels=[M5_NAMES[a] for a in M5_ARMS],title=sc.title())
cax=fig.add_axes([.91,.23,.018,.54]);fig.colorbar(im,cax=cax,label='Accuracy')
fig.suptitle('06  |  Arm accuracy by query type',fontsize=14,y=.96)
m5_save(fig,'06_arm_accuracy_by_question','Evaluator-only counterfactual outcomes. The online selector observed only its chosen-arm feedback.')


### 图7 · 突变前后描述
配对对话95%区间未做多重校正；不替代主检验。M4填写：全程平均是否掩盖阶段差异？

把解读写入 `deliverables/M5_notes.md` 并注明本次图表路径；数据未产生前不填结论。


In [ ]:
M5_PHASE_EFFECTS=[]
z=M5_REPLAY[M5_REPLAY.scenario=='abrupt'].copy();z['phase']=np.where(z.t<M5_CFG['T']//2,'before','after')
p=z.groupby(['seed','phase','method']).utility.mean().unstack('method')
fig,ax=plt.subplots(figsize=(10.6,5.4));fig.subplots_adjust(left=.11,right=.98,top=.77,bottom=.22)
for j,method in enumerate(['TS','LaplaceCTS','RuleRouter']):
    means=[];bounds=[]
    for phase in ['before','after']:
        g=p.xs(phase,level='phase');d=g[method]-g[M5_BEST['abrupt']];lo,hi=m5_boot_mean(d)
        mean=float(d.mean());means.append(mean);bounds.append([lo,hi])
        M5_PHASE_EFFECTS.append(dict(method=method,phase=phase,effect=mean,low=float(lo),high=float(hi),n=len(d),analysis='post-hoc descriptive, unadjusted 95% paired dialogue bootstrap'))
    xx=np.arange(2)+(j-1)*.23;bounds=np.array(bounds)
    ax.bar(xx,means,width=.21,color=M5_COLORS[method],label=method)
    for x0,(lo,hi) in zip(xx,bounds):ax.plot([x0,x0],[lo,hi],color='#23333E',lw=1.3)
    for x0,mean,(lo,hi) in zip(xx,means,bounds):ax.text(x0,hi+.007 if mean>=0 else lo-.009,f'{mean:+.4f}',ha='center',va='bottom' if mean>=0 else 'top',fontsize=10)
ax.axhline(0,color='#596874',lw=1);ax.set(xticks=[0,1],xticklabels=['Before shift: questions 1–150','After shift: questions 151–300'],
    ylabel=f"Utility difference vs {M5_NAMES[M5_BEST['abrupt']]}")
ax.margins(y=.18)
ax.grid(axis='y');ax.set_axisbelow(True)
fig.legend(*ax.get_legend_handles_labels(),loc='upper center',bbox_to_anchor=(.5,.89),ncol=3,frameon=False)
fig.suptitle('07  |  Paired utility differences before and after the question-mix shift',fontsize=14,y=.98)
pd.DataFrame(M5_PHASE_EFFECTS).to_csv(M5_DIR/'phase_effects_descriptive.csv',index=False)
m5_save(fig,'07_before_after_effects','Post-hoc descriptive 95% paired dialogue bootstrap CI; no family correction. Compare with the primary whole-dialogue test.')


In [ ]:
# 从实际CSV生成评价数值；归档图表清单与M4来源，保持M4目录不变。
M5_REPORT=['# M5 实验可视化与结果评价','',f'来源：`{M5_SOURCE}`。阶段：{M5_MODE}，每场景n={M5_N}。',
    '本次只新增图表和评价；slides与视频尚未制作。','',
    '| 场景 | 固定基线 | CTS效用差 | 校正97.5%区间 | CTS准确率 | 基线准确率 | CTS相对RuleRouter效用差 |',
    '|---|---|---:|---|---:|---:|---:|']
for r in M5_PRIMARY.itertuples():
    c=M5_SUMMARY.loc[(r.scenario,'LaplaceCTS')]; b=M5_SUMMARY.loc[(r.scenario,r.baseline)]; rr=M5_SUMMARY.loc[(r.scenario,'RuleRouter')]
    M5_REPORT.append(f'| {r.scenario} | {r.baseline} | {r.effect:+.5f} | [{r.family_low:.5f}, {r.family_high:.5f}] | {c.y:.2%} | {b.y:.2%} | {c.utility-rr.utility:+.5f} |')
M5_REPORT+=['','## 图表索引']
captions=['主次比较与区间','全方法效用/准确率/成本','收益来源分解','随时间的选臂比例','累计效用差及逐点区间','按问题类型的四臂准确率','突变前后效应（描述性）']
for name,caption in zip(M5_FIGURES,captions):M5_REPORT.append(f'- [{caption}]({name}.png)（另有同名SVG）')
M5_REPORT+=['','## 按本次数据判断（demo仅检查流程）']
for r in M5_PRIMARY.itertuples():
    verdict='支持正收益' if r.family_low>0 else ('支持负收益' if r.family_high<0 else '证据不足；不等于方法等效')
    M5_REPORT.append(f'- {r.scenario}：{verdict}；点估计≥0.02：{r.effect>=.02}，区间下界≥0.02：{r.family_low>=.02}。')
M5_REPORT+=['','## 仍需人工完成',
    '请在deliverables/M5_notes.md逐图写观察、证据、局限，由M1–M4复核；本表不代表slides/视频已经完成。',
    '限制：合成模板、规则回答器、即时无噪声反馈、proxy成本、全部候选构造开销。',
    'CTS与TS模型族不同，不能称为严格特征因果消融；图2/5/7区间仅描述。']
report='\n'.join(M5_REPORT)+'\n';(M5_DIR/'M5_results_evaluation.md').write_text(report,encoding='utf-8')
M5_NOTEBOOK=json.loads((M5_SOURCE/'notebook_snapshot.ipynb').read_text(encoding='utf-8'))
m5_cells=[c for c in M5_NOTEBOOK['cells'] if 'm5' in c.get('metadata',{}).get('tags',[])]
m5_source='\n\n'.join(''.join(c['source']) for c in m5_cells if c['cell_type']=='code')
(M5_DIR/'m5_source.py').write_text(m5_source,encoding='utf-8')
assert len(M5_FIGURES)==7
assert len(list(M5_DIR.glob('*.png')))==len(list(M5_DIR.glob('*.svg')))==7
for name,sha in M5_MANIFEST['files'].items():assert m5_sha(M5_SOURCE/name)==sha
M5_RECORD=dict(status='complete',stage='M5 visualization and evaluation only',
    created_at=datetime.now().astimezone().isoformat(),
    source_run=str(M5_SOURCE),source_manifest_sha256=m5_sha(M5_SOURCE/'manifest.json'),
    source_code_sha256=M5_MANIFEST['code_sha256'],m5_code_sha256=hashlib.sha256(m5_source.encode()).hexdigest(),
    mode=M5_MODE,n_dialogues_per_scenario=M5_N,figures=M5_FIGURES,
    png_count=7,svg_count=7,slides_or_video_created=False,original_archive_unchanged=True,
    runtime_seconds=time.perf_counter()-M5_STARTED,
    versions={'numpy':np.__version__,'pandas':pd.__version__,'matplotlib':matplotlib.__version__},
    checks=['source archive SHA256 before and after plotting','dialogue sample size',
            'utility decomposition identity','selection shares sum to one',
            'cumulative final gain agrees with primary effect','seven PNG/SVG pairs'],
    files={p.name:m5_sha(p) for p in sorted(M5_DIR.iterdir()) if p.is_file()})
(M5_DIR/'manifest_m5.json').write_text(json.dumps(M5_RECORD,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
display(Markdown(report))
print('M5 complete:',M5_DIR.resolve())


## 课程对应与完成标准
Lecture 2：Beta共轭、MAP与Laplace近似；Lecture 6：bandit、探索与后验采样；Lecture 4：开发/评估隔离；Lecture 1：统计建模与不确定性。课程原件不在本仓库，引用页码请各成员用自己的合法课件复核，不能把旧计划的页码称为本次已核查。

主线完成标准：真实双人10题审计 → 方法解释与数值检查 → 五人确认冻结口径 → 正式完整归档 → 七图与逐图解释 → slides/视频/贡献声明 → 全员签收。程序的COMPLETE只表示对应计算阶段完成。

可选DTS开发表已经保留；Markov/UCB仅保留实现供理解，不额外执行、不扩充主比较。遇到接口/参数更改先记录在PR并通知接收人，重新生成审查材料；不要只在自己的notebook副本里修正。
